In [1]:
import pandas as pd
import numpy as np

from sklearn.ensemble import IsolationForest

import folium

In [2]:
df = pd.read_csv(
    "../data/raw/ksp_synthetic_crime_dataset.csv"
)

print(df.shape)

df.head()

(10000, 11)


,crime_id,district,police_station,crime_type,date,latitude,longitude,victim_age,offender_age,status,repeat_offender
0,CR000001,Kalaburagi,Kalaburagi PS 4,Vehicle Theft,2023-09-28,17.362084,76.910451,56,36,Closed,0
1,CR000002,Ballari,Ballari PS 5,Assault,2025-05-21,15.110356,76.895142,38,50,Charge Sheet Filed,0
2,CR000003,Tumakuru,Tumakuru PS 1,Cybercrime,2023-02-28,13.314309,77.095126,33,32,Under Investigation,0
3,CR000004,Belagavi,Belagavi PS 4,Robbery,2023-09-01,15.822481,74.503246,68,24,Open,0
4,CR000005,Mysuru,Mysuru PS 4,Theft,2023-11-12,12.194464,76.695371,43,19,Charge Sheet Filed,0


In [3]:
features = df[
[
    'latitude',
    'longitude',
    'victim_age',
    'offender_age'
]
]

In [4]:
iso = IsolationForest(
    contamination=0.02,
    random_state=42
)

df['anomaly'] = iso.fit_predict(features)

In [5]:
df['anomaly'].value_counts()

anomaly
 1    9800
-1     200
Name: count, dtype: int64

In [6]:
anomalies = df[
    df['anomaly'] == -1
]

anomalies.head()

,crime_id,district,police_station,crime_type,date,latitude,longitude,victim_age,offender_age,status,repeat_offender,anomaly
5,CR000006,Kalaburagi,Kalaburagi PS 4,Vehicle Theft,2025-02-14,17.370827,76.773258,19,23,Under Investigation,0,-1
19,CR000020,Belagavi,Belagavi PS 2,Vehicle Theft,2024-11-14,15.947952,74.400704,72,21,Closed,0,-1
23,CR000024,Kalaburagi,Kalaburagi PS 2,Cybercrime,2023-09-20,17.381893,76.860622,74,19,Open,0,-1
344,CR000345,Bengaluru Urban,Bengaluru PS 1,Robbery,2024-07-17,13.033759,77.549659,74,64,Closed,0,-1
601,CR000602,Kalaburagi,Kalaburagi PS 2,Burglary,2025-02-13,17.289049,76.875464,18,21,Under Investigation,0,-1


In [7]:
anomalies.to_csv(
    "../data/processed/crime_anomalies.csv",
    index=False
)

In [8]:
m = folium.Map(
    location=[
        df.latitude.mean(),
        df.longitude.mean()
    ],
    zoom_start=7
)

In [9]:
for _, row in anomalies.iterrows():

    folium.CircleMarker(
        location=[
            row['latitude'],
            row['longitude']
        ],
        radius=5,
        popup=f"Crime ID: {row['crime_id']}",
        fill=True
    ).add_to(m)

In [10]:
m.save(
    "../gis/outputs/maps/crime_anomalies.html"
)

print("Anomaly map saved!")

Anomaly map saved!


In [11]:
from sklearn.ensemble import IsolationForest

features = df[
[
    'latitude',
    'longitude',
    'victim_age',
    'offender_age'
]
]

iso = IsolationForest(
    contamination=0.02,
    random_state=42
)

df['anomaly'] = iso.fit_predict(features)

In [12]:
df['anomaly'].value_counts()

anomaly
 1    9800
-1     200
Name: count, dtype: int64

In [13]:
anomalies = df[df['anomaly'] == -1]

anomalies[['crime_id',
           'district',
           'crime_type',
           'victim_age',
           'offender_age']].head(20)

,crime_id,district,crime_type,victim_age,offender_age
5,CR000006,Kalaburagi,Vehicle Theft,19,23
19,CR000020,Belagavi,Vehicle Theft,72,21
23,CR000024,Kalaburagi,Cybercrime,74,19
344,CR000345,Bengaluru Urban,Robbery,74,64
601,CR000602,Kalaburagi,Burglary,18,21
623,CR000624,Bengaluru Urban,Cybercrime,73,20
671,CR000672,Bengaluru Urban,Cybercrime,23,18
722,CR000723,Belagavi,Theft,18,22
733,CR000734,Belagavi,Theft,20,63
743,CR000744,Bengaluru Urban,Burglary,65,18


In [14]:
district_anomalies = (
    anomalies.groupby('district')
    .size()
    .reset_index(name='anomaly_count')
    .sort_values(
        by='anomaly_count',
        ascending=False
    )
)

district_anomalies.head(10)

,district,anomaly_count
4,Kalaburagi,51
1,Belagavi,45
5,Mangaluru,29
2,Bengaluru Urban,28
6,Mysuru,18
9,Vijayapura,15
3,Dharwad,7
7,Shivamogga,3
0,Ballari,2
8,Tumakuru,2


In [15]:
anomalies.to_csv(
    "../data/processed/crime_anomalies.csv",
    index=False
)

district_anomalies.to_csv(
    "../data/processed/district_anomalies.csv",
    index=False
)

In [16]:
import folium

m = folium.Map(
    location=[
        df.latitude.mean(),
        df.longitude.mean()
    ],
    zoom_start=7
)

for _, row in anomalies.iterrows():

    folium.CircleMarker(
        location=[
            row['latitude'],
            row['longitude']
        ],
        radius=5,
        popup=f"""
        Crime ID: {row['crime_id']}
        District: {row['district']}
        Crime: {row['crime_type']}
        """,
        fill=True
    ).add_to(m)

m

In [17]:
m.save(
    "../gis/outputs/maps/crime_anomalies.html"
)